# Bell State Simulator

**Day 22 | 30-Day Quantum Computing Challenge**

## Aim:
To prepare a two-qubit Bell state using Hadamard and CNOT gates and analyze correlated measurements.

## Tools:
Python, Qiskit, Qiskit Aer, Matplotlib

## Expected Result:
Approximately 50% outcomes 00 and 11, with no 01 or 10 outcomes in the ideal model.

**Execution:** Run the cells from top to bottom. The notebook uses an ideal simulator unless explicitly stated otherwise.

## 1. Theory and circuit

Starting with $|00\rangle$, apply $H$ to qubit 0 and then a controlled-NOT (control qubit 0, target qubit 1):

$|00\rangle \xrightarrow{H_0} (|00\rangle+|10\rangle)/\sqrt{2} \xrightarrow{CX_{0,1}} (|00\rangle+|11\rangle)/\sqrt{2}$.

When both qubits are measured in the computational (Z) basis, only **00** and **11** can occur in an ideal simulation. Correlated results in this one basis alone do **not** prove entanglement; verification requires additional measurements or a witness.

In [ ]:
# Experiment step: execute this cell and inspect its output.
from IPython.display import display, Image
display(Image(filename="circuit.png"))

## 2. Prepare the statevector (NumPy)

In [ ]:
# Experiment step: execute this cell and inspect its output.
import numpy as np
np.set_printoptions(precision=4, suppress=True)
# Statevector order: |00>, |01>, |10>, |11> (q0 is leftmost in this explanation)
zero = np.array([1, 0], dtype=complex)
H = np.array([[1, 1], [1, -1]], dtype=complex) / np.sqrt(2)
I = np.eye(2, dtype=complex)
CX = np.array([[1,0,0,0],[0,1,0,0],[0,0,0,1],[0,0,1,0]], dtype=complex)
initial = np.kron(zero, zero)
after_h = np.kron(H, I) @ initial
bell = CX @ after_h
print("Initial:", initial)
print("After H:", after_h)
print("Bell state:", bell)
print("Probabilities:", np.abs(bell)**2)
assert np.allclose(np.abs(bell)**2, [0.5, 0, 0, 0.5])

## 3. Run 1,000 shots and count joint outcomes

In [ ]:
# Experiment step: execute this cell and inspect its output.
import matplotlib.pyplot as plt
from collections import Counter
shots = 1000
seed = 22
rng = np.random.default_rng(seed)
labels = ["00", "01", "10", "11"]
probabilities = np.abs(bell)**2
samples = rng.choice(labels, size=shots, p=probabilities)
counts = {label: int(np.sum(samples == label)) for label in labels}
print("Counts:", counts)
print("Observed probability:", {k: round(v/shots, 4) for k,v in counts.items()})
assert counts["01"] == 0 and counts["10"] == 0
assert sum(counts.values()) == shots

## 4. Visualize the results

In [ ]:
# Experiment step: execute this cell and inspect its output.
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].bar(labels, [counts[x] for x in labels], color=["#314e91", "#a9bddf", "#a9bddf", "#7053a7"])
axes[0].set(title="Bell-state measurement outcomes", xlabel="Outcome", ylabel="Counts")
for i, key in enumerate(labels):
    axes[0].text(i, counts[key]+10, str(counts[key]), ha="center")
joint = np.array([[counts["00"],counts["01"]],[counts["10"],counts["11"]]]) / shots
axes[1].imshow(joint, cmap="Purples", vmin=0, vmax=0.6)
for i in range(2):
    for j in range(2):
        axes[1].text(j, i, f"{joint[i,j]:.1%}", ha="center", va="center")
axes[1].set(xticks=[0,1], yticks=[0,1], xticklabels=["q1=0","q1=1"], yticklabels=["q0=0","q0=1"], title="Joint probabilities")
fig.tight_layout()
fig.savefig("results.png", dpi=180, bbox_inches="tight")
plt.show()

## 5. Optional: Run the equivalent Qiskit circuit
Install in your notebook environment: `pip install qiskit qiskit-aer`. This section is optional and not required for the NumPy experiment. **Note:** Qiskit displays bitstrings in classical-register order; for this symmetric Bell state the observed labels are unchanged.

In [ ]:
# Uncomment after installing qiskit and qiskit-aer
# from qiskit import QuantumCircuit
# from qiskit_aer import AerSimulator
# qc = QuantumCircuit(2, 2)
# qc.h(0)
# qc.cx(0, 1)
# qc.measure([0, 1], [0, 1])
# print(qc.draw(output="text"))
# result = AerSimulator().run(qc, shots=1000, seed_simulator=22).result()
# print(result.get_counts())

## 6. What did we learn?
- The Hadamard gate creates a superposition; CNOT correlates the two qubits.
- Ideal Z-basis outcomes are **00** and **11** with 50% probability each; **01** and **10** have zero probability.
- Each qubit individually appears random, but their outcomes match.
- A simulation demonstrates the predicted distribution; a physical device may show errors and noise.
- **Challenge:** Apply an X gate to the second qubit after Bell preparation. Predict the new allowed outcomes before running the code.

## Observation:
Record the measured or simulated outputs and compare them with the theoretical expectation. Note that finite-shot results can fluctuate.

## Conclusion:
The experiment demonstrates the quantum circuit or state transformation described above using simulation; it does not establish performance on physical quantum hardware.